# %% [markdown]
# Condense names in SimpleStories/TinyStories dataset to ten canonical tokens

Stream‑only pipeline, no manual inspection:

1. Detect capitalised words that are **not** sentence‑initial.  
2. Keep tokens that appear in ≥ 20 stories.  
3. Remove obvious non‑names (weekdays, months, etc.).  
4. Cluster names that co‑occur **and** differ by Levenshtein ≤ 2 → choose one representative.  
5. Per‑story replacement so different real names never collide inside one story.  
6. Persist rewritten corpus, optionally drop stories with > 10 distinct names.


In [ ]:
# %%
# Imports
import re, itertools, gzip
from collections import Counter
import pandas as pd
import dask.dataframe as dd
import networkx as nx
import Levenshtein as lev


In [ ]:
# %%
CAP_RE   = re.compile(r"\b[A-Z][a-z]{2,30}\b")
SENT_END = re.compile(r"[.!?]")

def caps_not_sentence_start(txt: str):
    """Return capitalised tokens that are **not** the first word
    of a sentence."""
    out = []
    start = 0
    for m in SENT_END.finditer(txt):
        sent = txt[start:m.end()]
        out.extend(CAP_RE.findall(sent)[1:])      # skip first word
        start = m.end()
    out.extend(CAP_RE.findall(txt[start:])[1:])
    return out


In [ ]:
# %%
# Stream dataset once to build two counters
name_tok = Counter()   # total occurrences
name_doc = Counter()   # number of stories containing the token

PARQUET_PATH = "stories.parquet"   # adjust

ds = dd.read_parquet(PARQUET_PATH, columns=["text"])

for part in ds['text'].to_delayed():
    for story in part.compute():
        seen = set(caps_not_sentence_start(story))
        name_tok.update(seen)
        name_doc.update(seen)


In [ ]:
# %%
# Remove obvious non‑names and keep those appearing in ≥ 20 stories
STOP = {
    "Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday",
    "January","February","March","April","May","June","July","August",
    "September","October","November","December",
    "Dad","Mum","Mom","Father","Mother","Brother","Sister",
    "Love","Joy","Spring","Summer","Autumn","Winter"
}

candidates = {w for w in name_doc
              if name_doc[w] >= 20 and
                 w not in STOP and
                 w.lower() not in STOP}
print(f"{len(candidates):,} candidate names")


In [ ]:
# %%
# Cheap inverted index: bucket by first letter for speed
from collections import defaultdict
bucket = defaultdict(list)
for w in candidates:
    bucket[w[0].lower()].append(w)

# Build graph
G = nx.Graph()
G.add_nodes_from(candidates)

def cooccur(a: str, b: str) -> bool:
    """Return True if *a* and *b* appeared in the same story at least once.
    Very cheap sketch using name_doc from earlier pass. For real use,
    replace with a Bloom filter built during the counting pass."""
    return False   # placeholder – see note above

for names in bucket.values():
    for a, b in itertools.combinations(names, 2):
        if lev.distance(a.lower(), b.lower()) <= 2 and cooccur(a, b):
            G.add_edge(a, b)

alias_sets = list(nx.connected_components(G))
print(f"{len(alias_sets):,} alias sets")


In [ ]:
# %%
rep_of = {}
for s in alias_sets:
    root = max(s, key=name_tok.get)
    for a in s:
        rep_of[a] = root


In [ ]:
# %%
CANON = ["Alex","Bailey","Casey","Drew","Eden",
         "Fin","Gray","Hayden","Jordan","Quinn"]


In [ ]:
# %%
pattern = re.compile(r"\b(" + "|".join(map(re.escape, rep_of.keys())) + r")\b")

def rewrite(story: str):
    mapping = {}
    def repl(m):
        orig = m.group(1)
        root = rep_of.get(orig, orig)
        if root not in mapping:
            mapping[root] = CANON[len(mapping) % 10]
        return mapping[root]
    return pattern.sub(repl, story)


In [ ]:
# %%
ddf = dd.read_parquet(PARQUET_PATH, columns=["id","text"])
ddf["text"] = ddf["text"].map_partitions(lambda s: s.apply(rewrite),
                                         meta=("text","object"))
ddf.to_parquet("stories_names10.parquet")


In [ ]:
# %%
def too_many(s: str):
    return sum(1 for _ in pattern.finditer(s)) > 10

ddf = ddf[~ddf["text"].map_partitions(lambda s: s.apply(too_many),
                                      meta=("x","bool"))]
ddf.to_parquet("stories_names10_clean.parquet")


In [ ]:
# %%
from collections import Counter
name_counter = Counter()

for part in dd.read_parquet("stories_names10_clean.parquet", columns=["text"])["text"].to_delayed():
    for story in part.compute():
        name_counter.update(re.findall(r"\b(" + "|".join(CANON) + r")\b", story))

print(name_counter.most_common())
